In [1]:
MATCH = 1
MISMATCH = -1
GAP = -2

In [2]:
def score(a, b):
    return MATCH if a == b else MISMATCH


def build_matrix(s1, s2):
    rows, cols = len(s1) + 1, len(s2) + 1
    M = [[0] * cols for _ in range(rows)]
    T = [[""] * cols for _ in range(rows)]
    T[0][0] = "*"
    for i in range(1, rows):
        M[i][0] = M[i - 1][0] + GAP
        T[i][0] = "U"
    for j in range(1, cols):
        M[0][j] = M[0][j - 1] + GAP
        T[0][j] = "L"
    for i in range(1, rows):
        for j in range(1, cols):
            diag = M[i - 1][j - 1] + score(s1[i - 1], s2[j - 1])
            up = M[i - 1][j] + GAP
            left = M[i][j - 1] + GAP
            best = max(diag, up, left)
            M[i][j] = best
            if best == diag:
                T[i][j] = "D"
            elif best == up:
                T[i][j] = "U"
            else:
                T[i][j] = "L"
    return M, T

In [3]:
def traceback(s1, s2, M, T):
    a1, a2 = [], []
    i, j = len(s1), len(s2)
    path = [(i, j)]
    steps = []
    while i > 0 or j > 0:
        move = T[i][j]
        if move == "D":
            x, y = s1[i - 1], s2[j - 1]
            kind = "match" if x == y else "mismatch"
            steps.append((i, j, M[i][j], "diagonal", f"{x} / {y}", kind))
            a1.append(x); a2.append(y)
            i -= 1; j -= 1
        elif move == "U":
            x = s1[i - 1]
            steps.append((i, j, M[i][j], "up", f"{x} / -", "gap in S2"))
            a1.append(x); a2.append("-")
            i -= 1
        else:
            y = s2[j - 1]
            steps.append((i, j, M[i][j], "left", f"- / {y}", "gap in S1"))
            a1.append("-"); a2.append(y)
            j -= 1
        path.append((i, j))
    return "".join(reversed(a1)), "".join(reversed(a2)), path[::-1], steps

In [4]:
ARROWS = {"D": "\u2196", "U": "\u2191", "L": "\u2190", "*": "\u2022"}


def print_matrix(s1, s2, M, path=None):
    on_path = set(path or [])
    print("        " + "".join(f"{c:>6}" for c in "-" + s2))
    for i, row in enumerate(M):
        label = "-" if i == 0 else s1[i - 1]
        cells = ""
        for j, v in enumerate(row):
            cells += f"{'[' + str(v) + ']':>6}" if (i, j) in on_path else f"{v:>6}"
        print(f"   {label:>3}  " + cells)


def print_traceback_matrix(s1, s2, T, path=None):
    on_path = set(path or [])
    print("        " + "".join(f"{c:>6}" for c in "-" + s2))
    for i, row in enumerate(T):
        label = "-" if i == 0 else s1[i - 1]
        cells = ""
        for j, t in enumerate(row):
            a = ARROWS[t]
            cells += f"{'[' + a + ']':>6}" if (i, j) in on_path else f"{a:>6}"
        print(f"   {label:>3}  " + cells)


def print_steps(steps):
    print(f"   {'step':>4}  {'cell':>8}  {'score':>5}  {'move':<9} {'pair':<6} type")
    for n, (i, j, v, move, pair, kind) in enumerate(steps, 1):
        print(f"   {n:>4}  {f'({i},{j})':>8}  {v:>5}  {move:<9} {pair:<6} {kind}")


def match_line(a1, a2):
    return "".join("|" if x == y else (" " if "-" in (x, y) else ".")
                   for x, y in zip(a1, a2))


def global_align(s1, s2):
    M, T = build_matrix(s1, s2)
    a1, a2, path, steps = traceback(s1, s2, M, T)
    return M, T, a1, a2, path, steps, M[-1][-1]

In [5]:
s1 = "ATCCGTTAC"
s2 = "CGAT"

print(f"S1 = {s1}  (length {len(s1)})")
print(f"S2 = {s2}  (length {len(s2)})")
print(f"Scoring: match={MATCH}, mismatch={MISMATCH}, gap={GAP}\n")

M, T, a1, a2, path, steps, final = global_align(s1, s2)

print("Score matrix (traceback path in [ ]):")
print_matrix(s1, s2, M, path)

print("\nTraceback matrix (\u2196 diagonal, \u2191 up, \u2190 left, path in [ ]):")
print_traceback_matrix(s1, s2, T, path)

print("\nTraceback steps (from bottom-right to top-left):")
print_steps(steps)

print("\nPath:", " -> ".join(f"({i},{j})" for i, j in path))

S1 = ATCCGTTAC  (length 9)
S2 = CGAT  (length 4)
Scoring: match=1, mismatch=-1, gap=-2

Score matrix (traceback path in [ ]):
             -     C     G     A     T
     -     [0]    -2    -4    -6    -8
     A    [-2]    -1    -3    -3    -5
     T    [-4]    -3    -2    -4    -2
     C    [-6]    -3    -4    -3    -4
     C      -8  [-5]    -4    -5    -4
     G     -10    -7  [-4]    -5    -6
     T     -12    -9  [-6]    -5    -4
     T     -14   -11  [-8]    -7    -4
     A     -16   -13   -10  [-7]    -6
     C     -18   -15   -12    -9  [-8]

Traceback matrix (↖ diagonal, ↑ up, ← left, path in [ ]):
             -     C     G     A     T
     -     [•]     ←     ←     ←     ←
     A     [↑]     ↖     ↖     ↖     ←
     T     [↑]     ↖     ↖     ↖     ↖
     C     [↑]     ↖     ↖     ↖     ↑
     C       ↑   [↖]     ↖     ↖     ↖
     G       ↑     ↑   [↖]     ↖     ↖
     T       ↑     ↑   [↑]     ↖     ↖
     T       ↑     ↑   [↑]     ↖     ↖
     A       ↑     ↑     ↑   [↖]   

In [6]:
matches = sum(x == y for x, y in zip(a1, a2))
gaps = a1.count("-") + a2.count("-")

print("Optimal global alignment:")
print("   S1: " + a1)
print("       " + match_line(a1, a2))
print("   S2: " + a2)
print(f"\nAlignment score : {final}")
print(f"Matches         : {matches}")
print(f"Gaps            : {gaps}")
print(f"Identity        : {matches}/{len(a1)} = {100 * matches / len(a1):.1f}%")

Optimal global alignment:
   S1: ATCCGTTAC
          ||  |.
   S2: ---CG--AT

Alignment score : -8
Matches         : 3
Gaps            : 5
Identity        : 3/9 = 33.3%
